In [75]:
import pandas as pd

In [85]:
# Path to GLSEA SST data
usace_file = '/Users/ljob/Desktop/Data/USACE/GLSHFSModel_merged.csv'
nbs_p_file = '/Users/ljob/Desktop/CNBS_forecast_ver_swe_sst.csv'

#ver_file = 'CNBS_forecast_ver.csv'
#bias_file = 'CNBS_forecast_ver_swe.csv'

existing_model_data = pd.read_csv(usace_file,sep='\t')
my_model_data = pd.read_csv(nbs_p_file,sep='\t')


In [77]:
import pandas as pd
import numpy as np


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def standardize_existing_nbs_model(
    existing_df,
    date_col="forecast_date",
    month_col="year_month",
    model_name="existing_model"
):
    df = existing_df.copy()

    df = df.rename(columns={
        date_col: "cfs_run",
        month_col: "forecast_month"
    })

    df["cfs_run"] = pd.to_datetime(df["cfs_run"])
    df["forecast_month"] = pd.to_datetime(df["forecast_month"])
    df["existing_model"] = model_name

    return df


def melt_my_nbs_model(my_df):
    df = my_df.copy()

    df["cfs_run"] = pd.to_datetime(df["cfs_run"])
    df["forecast_month"] = pd.to_datetime(df["forecast_month"])

    id_cols = ["cfs_run", "forecast_month", "model"]

    # only compare NBS forecast + NBS obs
    nbs_cols = [
        c for c in df.columns
        if c.endswith("_nbs") or c.endswith("_nbs_obs")
    ]

    long = df.melt(
        id_vars=id_cols,
        value_vars=nbs_cols,
        var_name="variable",
        value_name="value"
    )

    long["type"] = np.where(
        long["variable"].str.endswith("_obs"),
        "obs",
        "forecast"
    )

    long["variable"] = long["variable"].str.replace("_obs", "", regex=False)

    split = long["variable"].str.split("_", n=1, expand=True)
    long["lake"] = split[0]
    long["component"] = split[1]

    wide = (
        long
        .pivot_table(
            index=["cfs_run", "forecast_month", "model", "lake", "component"],
            columns="type",
            values="value"
        )
        .reset_index()
    )

    return wide


def melt_existing_nbs_model(existing_df):
    df = existing_df.copy()

    id_cols = ["cfs_run", "forecast_month", "existing_model"]

    nbs_cols = [
        c for c in df.columns
        if c.endswith("_nbs")
    ]

    long = df.melt(
        id_vars=id_cols,
        value_vars=nbs_cols,
        var_name="variable",
        value_name="existing_forecast"
    )

    split = long["variable"].str.split("_", n=1, expand=True)
    long["lake"] = split[0]
    long["component"] = split[1]

    return long.drop(columns="variable")

def compare_my_model_to_existing_nbs(
    my_df,
    existing_df,
    existing_model_name="existing_model"
):
    """
    Positive improvement_pct means your model has lower RMSE.

    Example:
    +15 means your model is 15% better than the existing model.
    -10 means your model is 10% worse than the existing model.
    """

    existing_standard = standardize_existing_nbs_model(
        existing_df,
        model_name=existing_model_name
    )

    my_long = melt_my_nbs_model(my_df)
    existing_long = melt_existing_nbs_model(existing_standard)

    merged = my_long.merge(
        existing_long,
        on=["cfs_run", "forecast_month", "lake", "component"],
        how="inner"
    )

    merged = merged.dropna(subset=["forecast", "obs", "existing_forecast"])

    # lake/component breakdown
    comparison = (
        merged
        .groupby(["model", "lake", "component"])
        .apply(
            lambda g: pd.Series({
                "my_rmse": rmse(g["obs"], g["forecast"]),
                "existing_rmse": rmse(g["obs"], g["existing_forecast"]),
            })
        )
        .reset_index()
    )

    comparison["improvement_pct"] = (
        (comparison["existing_rmse"] - comparison["my_rmse"])
        / comparison["existing_rmse"]
        * 100
    )

    comparison["winner"] = np.where(
        comparison["improvement_pct"] > 0,
        "my_model",
        existing_model_name
    )

    comparison = comparison.sort_values("improvement_pct", ascending=False)

    # overall by model only
    overall = (
        merged
        .groupby("model")
        .apply(
            lambda g: pd.Series({
                "my_rmse": rmse(g["obs"], g["forecast"]),
                "existing_rmse": rmse(g["obs"], g["existing_forecast"]),
            })
        )
        .reset_index()
    )

    overall["improvement_pct"] = (
        (overall["existing_rmse"] - overall["my_rmse"])
        / overall["existing_rmse"]
        * 100
    )

    overall["winner"] = np.where(
        overall["improvement_pct"] > 0,
        "my_model",
        existing_model_name
    )

    overall = overall.sort_values("improvement_pct", ascending=False)

    return comparison, overall

In [78]:
comparison, overall = compare_my_model_to_existing_nbs(
    my_df=my_model_data,
    existing_df=existing_model_data,
    existing_model_name="existing_model"
)

print("\n=== NBS by Model/Lake ===")
print(comparison)

print("\n=== Overall NBS by Model ===")
print(overall)


=== NBS by Model/Lake ===
   model            lake component     my_rmse  existing_rmse  \
8     RF            erie       nbs  106.816341     110.257696   
10    RF         ontario       nbs  110.340043     112.606889   
11    RF        superior       nbs   48.955017      48.291734   
9     RF  michigan-huron       nbs   48.415517      45.977396   
0     GP            erie       nbs  131.568679     110.257696   
15   XGB        superior       nbs   58.813365      48.291734   
3     GP        superior       nbs   59.528204      48.291734   
6     NN         ontario       nbs  139.112336     112.606889   
2     GP         ontario       nbs  139.432347     112.606889   
7     NN        superior       nbs   60.509246      48.291734   
12   XGB            erie       nbs  138.396088     110.257696   
4     NN            erie       nbs  139.360607     110.257696   
1     GP  michigan-huron       nbs   63.608458      45.977396   
5     NN  michigan-huron       nbs   64.095151      45.977396  

/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_18912/2255127864.py:131: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_18912/2255127864.py:158: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [79]:
print("\n=== Overall NBS by Model ===")
print(overall)


=== Overall NBS by Model ===
  model     my_rmse  existing_rmse  improvement_pct          winner
2    RF   84.150627      85.561503         1.648962        my_model
0    GP  105.286917      85.561503       -23.054076  existing_model
1    NN  107.869340      85.561503       -26.072283  existing_model
3   XGB  115.607269      85.561503       -35.115986  existing_model


In [80]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(comparison)

   model            lake component     my_rmse  existing_rmse  \
8     RF            erie       nbs  106.816341     110.257696   
10    RF         ontario       nbs  110.340043     112.606889   
11    RF        superior       nbs   48.955017      48.291734   
9     RF  michigan-huron       nbs   48.415517      45.977396   
0     GP            erie       nbs  131.568679     110.257696   
15   XGB        superior       nbs   58.813365      48.291734   
3     GP        superior       nbs   59.528204      48.291734   
6     NN         ontario       nbs  139.112336     112.606889   
2     GP         ontario       nbs  139.432347     112.606889   
7     NN        superior       nbs   60.509246      48.291734   
12   XGB            erie       nbs  138.396088     110.257696   
4     NN            erie       nbs  139.360607     110.257696   
1     GP  michigan-huron       nbs   63.608458      45.977396   
5     NN  michigan-huron       nbs   64.095151      45.977396   
14   XGB         ontario 

In [86]:
import pandas as pd
import numpy as np


# ----------------------------
# METRICS
# ----------------------------
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    if ss_tot == 0:
        return np.nan

    return 1 - (ss_res / ss_tot)


def bias(y_true, y_pred):
    # positive = model overpredicts
    # negative = model underpredicts
    return np.mean(y_pred - y_true)


def calc_metrics(g):
    return pd.Series({
        "my_rmse": rmse(g["obs"], g["forecast"]),
        "existing_rmse": rmse(g["obs"], g["existing_forecast"]),

        "my_r2": r2_score(g["obs"], g["forecast"]),
        "existing_r2": r2_score(g["obs"], g["existing_forecast"]),

        "my_bias": bias(g["obs"], g["forecast"]),
        "existing_bias": bias(g["obs"], g["existing_forecast"]),
    })


def add_improvement_columns(df, existing_model_name="existing_model"):
    df = df.copy()

    # RMSE: lower is better
    df["rmse_improvement_pct"] = (
        (df["existing_rmse"] - df["my_rmse"])
        / df["existing_rmse"]
        * 100
    )

    # R2: higher is better
    df["r2_improvement"] = df["my_r2"] - df["existing_r2"]

    # Bias: closer to zero is better
    df["bias_improvement_pct"] = np.where(
        np.abs(df["existing_bias"]) == 0,
        np.nan,
        (
            (np.abs(df["existing_bias"]) - np.abs(df["my_bias"]))
            / np.abs(df["existing_bias"])
            * 100
        )
    )

    df["rmse_winner"] = np.where(
        df["rmse_improvement_pct"] > 0,
        "my_model",
        existing_model_name
    )

    df["r2_winner"] = np.where(
        df["r2_improvement"] > 0,
        "my_model",
        existing_model_name
    )

    df["bias_winner"] = np.where(
        df["bias_improvement_pct"] > 0,
        "my_model",
        existing_model_name
    )

    return df


# ----------------------------
# STANDARDIZE EXISTING MODEL
# ----------------------------
def standardize_existing_nbs_model(
    existing_df,
    date_col="forecast_date",
    month_col="year_month",
    model_name="existing_model"
):
    df = existing_df.copy()

    df = df.rename(columns={
        date_col: "cfs_run",
        month_col: "forecast_month"
    })

    df["cfs_run"] = pd.to_datetime(df["cfs_run"])
    df["forecast_month"] = pd.to_datetime(df["forecast_month"])
    df["existing_model"] = model_name

    return df


# ----------------------------
# MELT YOUR MODEL DATA
# ----------------------------
def melt_my_nbs_model(my_df):
    df = my_df.copy()

    df["cfs_run"] = pd.to_datetime(df["cfs_run"])
    df["forecast_month"] = pd.to_datetime(df["forecast_month"])

    id_cols = ["cfs_run", "forecast_month", "model"]

    nbs_cols = [
        c for c in df.columns
        if c.endswith("_nbs") or c.endswith("_nbs_obs")
    ]

    long = df.melt(
        id_vars=id_cols,
        value_vars=nbs_cols,
        var_name="variable",
        value_name="value"
    )

    long["type"] = np.where(
        long["variable"].str.endswith("_obs"),
        "obs",
        "forecast"
    )

    long["variable"] = long["variable"].str.replace("_obs", "", regex=False)

    split = long["variable"].str.split("_", n=1, expand=True)
    long["lake"] = split[0]
    long["component"] = split[1]

    wide = (
        long
        .pivot_table(
            index=["cfs_run", "forecast_month", "model", "lake", "component"],
            columns="type",
            values="value"
        )
        .reset_index()
    )

    return wide


# ----------------------------
# MELT EXISTING MODEL DATA
# ----------------------------
def melt_existing_nbs_model(existing_df):
    df = existing_df.copy()

    id_cols = ["cfs_run", "forecast_month", "existing_model"]

    nbs_cols = [
        c for c in df.columns
        if c.endswith("_nbs")
    ]

    long = df.melt(
        id_vars=id_cols,
        value_vars=nbs_cols,
        var_name="variable",
        value_name="existing_forecast"
    )

    split = long["variable"].str.split("_", n=1, expand=True)
    long["lake"] = split[0]
    long["component"] = split[1]

    return long.drop(columns="variable")


# ----------------------------
# COMPARE YOUR MODEL TO EXISTING NBS MODEL
# ----------------------------
def compare_my_model_to_existing_nbs(
    my_df,
    existing_df,
    existing_model_name="existing_model"
):
    """
    Compares your NBS forecasts to an existing NBS-only model.

    Positive rmse_improvement_pct:
        Your model has lower RMSE.

    Positive r2_improvement:
        Your model has higher R2.

    Positive bias_improvement_pct:
        Your model has bias closer to zero.
    """

    existing_standard = standardize_existing_nbs_model(
        existing_df,
        model_name=existing_model_name
    )

    my_long = melt_my_nbs_model(my_df)
    existing_long = melt_existing_nbs_model(existing_standard)

    merged = my_long.merge(
        existing_long,
        on=["cfs_run", "forecast_month", "lake", "component"],
        how="inner"
    )

    merged = merged.dropna(subset=["forecast", "obs", "existing_forecast"])

    comparison = (
        merged
        .groupby(["model", "lake", "component"])
        .apply(calc_metrics)
        .reset_index()
    )

    comparison = add_improvement_columns(
        comparison,
        existing_model_name=existing_model_name
    )

    comparison = comparison.sort_values(
        "rmse_improvement_pct",
        ascending=False
    )

    overall = (
        merged
        .groupby("model")
        .apply(calc_metrics)
        .reset_index()
    )

    overall = add_improvement_columns(
        overall,
        existing_model_name=existing_model_name
    )

    overall = overall.sort_values(
        "rmse_improvement_pct",
        ascending=False
    )

    return comparison, overall


# ----------------------------
# USAGE
# ----------------------------
comparison, overall = compare_my_model_to_existing_nbs(
    my_df=my_model_data,
    existing_df=existing_model_data,
    existing_model_name="existing_model"
)

print("\n=== NBS Skill by Model/Lake/Component ===")
print(comparison)

print("\n=== Overall NBS Skill by Model ===")
print(overall)


=== NBS Skill by Model/Lake/Component ===
   model            lake component     my_rmse  existing_rmse     my_r2  \
11    RF        superior       nbs   53.279792      56.586432  0.587637   
9     RF  michigan-huron       nbs   40.259473      41.767628  0.648102   
3     GP        superior       nbs   54.686752      56.586432  0.565571   
7     NN        superior       nbs   58.834471      56.586432  0.497173   
15   XGB        superior       nbs   62.356378      56.586432  0.435172   
10    RF         ontario       nbs   99.990413      89.863846  0.330717   
1     GP  michigan-huron       nbs   47.194285      41.767628  0.516430   
8     RF            erie       nbs  118.684654     101.012999  0.065932   
0     GP            erie       nbs  121.583662     101.012999  0.019743   
2     GP         ontario       nbs  109.805797      89.863846  0.192870   
13   XGB  michigan-huron       nbs   54.726065      41.767628  0.349768   
12   XGB            erie       nbs  133.160868     101.01

/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_18912/157902155.py:224: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_metrics)
/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_18912/157902155.py:241: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_metrics)


In [87]:
print("\n=== Overall NBS Skill by Model ===")
print(overall)


=== Overall NBS Skill by Model ===
  model     my_rmse  existing_rmse     my_r2  existing_r2    my_bias  \
2    RF   84.474391      76.199895  0.411897     0.521467  25.490560   
0    GP   89.523518      76.199895  0.339493     0.521467  21.114277   
3   XGB  104.579836      76.199895  0.098639     0.521467  29.407369   
1    NN  119.847935      76.199895 -0.183761     0.521467  28.493917   

   existing_bias  rmse_improvement_pct  r2_improvement  bias_improvement_pct  \
2       2.106958            -10.858934       -0.109570          -1109.827873   
0       2.106958            -17.485094       -0.181974           -902.121582   
3       2.106958            -37.244068       -0.422828          -1295.726675   
1       2.106958            -57.280972       -0.705228          -1252.372599   

      rmse_winner       r2_winner     bias_winner  
2  existing_model  existing_model  existing_model  
0  existing_model  existing_model  existing_model  
3  existing_model  existing_model  existing_mo